In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Đường dẫn tương đối từ vị trí thư mục notebooks/
RAW_DATA_PATH = Path("../data/raw/hotel_bookings.csv")
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Kích thước ban đầu: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
df_raw.head()

Kích thước ban đầu: 119,390 dòng, 32 cột


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
# Kiểm tra các cột bị thiếu giá trị (Missing values)
missing_info = df_raw.isna().sum()[df_raw.isna().sum() > 0]
print("--- Danh sách cột bị thiếu dữ liệu ---")
print(missing_info)

# Phân phối tỷ lệ hủy phòng (is_canceled)
print("\n--- Tỷ lệ nhãn mục tiêu is_canceled ---")
print(df_raw['is_canceled'].value_counts(normalize=True).map('{:.2%}'.format))

--- Danh sách cột bị thiếu dữ liệu ---
children         4
country        488
agent        16340
company     112593
dtype: int64

--- Tỷ lệ nhãn mục tiêu is_canceled ---
is_canceled
0    62.96%
1    37.04%
Name: proportion, dtype: str


In [5]:
df_clean = df_raw.copy()

# 1. Bẫy lượt đặt 0 người: Xóa các đơn không có cả người lớn, trẻ em lẫn em bé
zero_guests_mask = (df_clean['adults'] == 0) & (df_clean['children'] == 0) & (df_clean['babies'] == 0)
print(f"Số dòng 0 người ở bị loại bỏ: {zero_guests_mask.sum()}")
df_clean = df_clean[~zero_guests_mask]

# 2. Bẫy giá phòng ADR: Loại bỏ ADR âm (< 0) và ngoại lai cực đại (adr >= 5000)
invalid_adr_mask = (df_clean['adr'] < 0) | (df_clean['adr'] >= 5000)
print(f"Số dòng ADR bất thường bị loại bỏ: {invalid_adr_mask.sum()}")
df_clean = df_clean[~invalid_adr_mask]

# 3. Điền giá trị thiếu (Missing values) đúng bản chất nghiệp vụ
df_clean['children'] = df_clean['children'].fillna(0)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)      # 0 mang nghĩa đặt trực tiếp
df_clean['company'] = df_clean['company'].fillna(0)    # 0 mang nghĩa khách cá nhân

print(f"Kích thước sau khi sửa bẫy dữ liệu: {df_clean.shape[0]:,} dòng")

Số dòng 0 người ở bị loại bỏ: 180
Số dòng ADR bất thường bị loại bỏ: 2
Kích thước sau khi sửa bẫy dữ liệu: 119,208 dòng


In [6]:
# Bắt buộc loại bỏ 2 cột rò rỉ đáp án trực tiếp
leakage_cols = ['reservation_status', 'reservation_status_date']
df_clean = df_clean.drop(columns=leakage_cols)

print(f"Số cột còn lại sau khi chống rò rỉ nhãn: {df_clean.shape[1]} cột")

Số cột còn lại sau khi chống rò rỉ nhãn: 30 cột


In [7]:
# Kiểm tra số lượng dòng trùng lặp hoàn toàn
num_duplicates = df_clean.duplicated().sum()
pct_duplicates = (num_duplicates / len(df_clean)) * 100
print(f"Số dòng trùng hoàn toàn: {num_duplicates:,} ({pct_duplicates:.2f}%)")

# Tạo thư mục data/processed nếu chưa có và xuất file sạch
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH = PROCESSED_DIR / "hotel_bookings_clean.csv"
df_clean.to_csv(CLEAN_DATA_PATH, index=False)

print(f"Đã xuất dữ liệu sạch thành công tại: {CLEAN_DATA_PATH}")

Số dòng trùng hoàn toàn: 32,238 (27.04%)
Đã xuất dữ liệu sạch thành công tại: ../data/processed/hotel_bookings_clean.csv
